In [ ]:
!pip install -q gymnasium

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Create FrozenLake-v1 Environment


env = gym.make(
    "FrozenLake-v1",
    is_slippery=True,
    render_mode=None
)

print("Environment created successfully!")
print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)

# Number of states and actions
n_states = env.observation_space.n
n_actions = env.action_space.n

In [ ]:
# Define TD(0) Parameters

gamma = 0.99      # Discount factor
alpha = 0.1       # Learning rate
epsilon = 0.1     # Exploration probability

episodes = 5000

print("\nTD(0) Parameters:")
print("Gamma   :", gamma)
print("Alpha   :", alpha)
print("Epsilon :", epsilon)
print("Episodes:", episodes)

In [ ]:
# Initialize State-Value Function

V = np.zeros(n_states)

# The goal state (state 15) has a known value of 1.0
V[15] = 1.0

print("\nInitial State-Value Function:")
print(V)


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# Epsilon-Greedy Action Selection

def epsilon_greedy_action(env, state, V, epsilon):
    """
    Select an action using epsilon-greedy policy.

    With probability epsilon:
        choose a random action.

    Otherwise:
        choose the action leading to the highest
        estimated value of the next state.
    """

    # Exploration
    if np.random.random() < epsilon:
        return env.action_space.sample()

    # Exploitation
    action_values = []

    for action in range(n_actions):

        # Obtain possible transitions from environment
        transitions = env.unwrapped.P[state][action]

        expected_value = 0

        for prob, next_s, reward, terminated_transition in transitions:
            if terminated_transition: # If next_s is a terminal state in this specific transition
                # If it's a terminal state, its value is the immediate reward it yields
                expected_value += prob * reward
            else:
                # Otherwise, it's the discounted value of the next state
                expected_value += prob * V[next_s]

        action_values.append(expected_value)

    return np.argmax(action_values)

In [ ]:
# TD(0) Algorithm

episode_rewards = []
value_history = []

for episode in range(episodes):

    # Reset environment
    state, info = env.reset()

    total_reward = 0
    done = False

    while not done:

        # Select action using epsilon-greedy policy
        action = epsilon_greedy_action(
            env,
            state,
            V,
            epsilon
        )

        # Take action
        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        # TD(0) Update
        #
        # TD Target = R + gamma * V(S')
        #
        # TD Error = R + gamma*V(S') - V(S)
        #
        # V(S) = V(S) + alpha * TD Error

        if terminated:
            td_target = reward
        else:
            td_target = reward + gamma * V[next_state]

        td_error = td_target - V[state]

        V[state] = V[state] + alpha * td_error

        # Move to next state
        state = next_state

        total_reward += reward

    episode_rewards.append(total_reward)

    # Store values periodically
    if episode % 100 == 0:
        value_history.append(V.copy())


print("\nTraining completed successfully!")

In [ ]:
#  Display Learned State Values

print("\nFinal State-Value Function:")
print(np.round(V, 4))

# Reshape into 4x4 FrozenLake grid
V_grid = V.reshape(4, 4)

print("\n4x4 State-Value Grid:")
print(np.round(V_grid, 4))


In [ ]:
# Plot State Values
plt.figure(figsize=(8, 6))

plt.imshow(V_grid)

plt.colorbar(label="State Value V(s)")

plt.title("TD(0) Estimated State Values - FrozenLake-v1")
plt.xlabel("Column")
plt.ylabel("Row")

# Display values inside cells
for i in range(4):
    for j in range(4):
        plt.text(
            j,
            i,
            f"{V_grid[i, j]:.3f}",
            ha="center",
            va="center"
        )

plt.xticks(range(4))
plt.yticks(range(4))

plt.show()

In [ ]:
# Plot Learning Curve

# Calculate moving average of rewards
window = 100

moving_average = np.convolve(
    episode_rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 5))

plt.plot(moving_average)

plt.title("TD(0) Learning Curve")
plt.xlabel("Episode")
plt.ylabel("Average Reward")

plt.grid(True)

plt.show()

In [ ]:
# Display FrozenLake Map

print("\nFrozenLake Environment:")

for row in env.unwrapped.desc:
    print(" ".join(
        cell.decode("utf-8") if isinstance(cell, bytes) else str(cell)
        for cell in row
    ))

In [ ]:
# State Value Table

print("\nState Value Table")
print("-" * 35)

for state in range(n_states):
    row = state // 4
    col = state % 4

    print(
        f"State {state:2d} "
        f"(row={row}, col={col}) : "
        f"V(s) = {V[state]:.4f}"
    )

In [ ]:
# Evaluate the Learned Value Function

evaluation_episodes = 1000

evaluation_rewards = []

for episode in range(evaluation_episodes):

    state, info = env.reset()

    total_reward = 0
    done = False

    while not done:

        # Greedy action selection
        action = epsilon_greedy_action(
            env,
            state,
            V,
            epsilon=0
        )

        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        total_reward += reward

        state = next_state

    evaluation_rewards.append(total_reward)


average_reward = np.mean(evaluation_rewards)

print("\nEvaluation Results")
print("-" * 35)
print("Evaluation Episodes :", evaluation_episodes)
print("Average Reward     :", round(average_reward, 4))
print("Successful Episodes:",
      sum(evaluation_rewards))
print(
    "Success Rate       :",
    round(average_reward * 100, 2),
    "%"
)


In [ ]:
#  Close Environment

env.close()

print("\nExperiment 7 completed successfully!")